In [ ]:

import sys
sys.path.insert(0, '/Users/arshgarg/Drug_Deepfake/cosyvoice')
sys.path.insert(0, '/Users/arshgarg/Drug_Deepfake/cosyvoice/third_party/Matcha-TTS')

# !pip install torch torchaudio librosa numpy scipy --quiet

print("✓ Dependencies installed")

✓ Dependencies installed


In [11]:
import os
import subprocess
from pathlib import Path

# Clone official CosyVoice repository
cosyvoice_dir = Path('/Users/arshgarg/Drug_Deepfake/CosyVoice') if '/content/' in os.getcwd() else Path('./CosyVoice')

if not cosyvoice_dir.exists():
    subprocess.run(['git', 'clone', '--recursive', 'https://github.com/FunAudioLLM/CosyVoice.git', str(cosyvoice_dir)])
    print(f"✓ Cloned CosyVoice to {cosyvoice_dir}")
else:
    print(f"✓ CosyVoice already exists at {cosyvoice_dir}")

# Install CosyVoice package
# Repository already cloned.
# Add it directly to Python path.

import sys

sys.path.insert(0, str(cosyvoice_dir))
sys.path.insert(0, str(cosyvoice_dir / "third_party" / "Matcha-TTS"))

print("✓ CosyVoice added to Python path")

✓ CosyVoice already exists at CosyVoice
✓ CosyVoice added to Python path


In [12]:
import sys
from pathlib import Path
import os

# Add CosyVoice to path
cosyvoice_dir = Path('/content/CosyVoice') if '/content/' in os.getcwd() else Path('./CosyVoice')
sys.path.insert(0, str(cosyvoice_dir))
sys.path.insert(0, str(cosyvoice_dir / 'third_party' / 'Matcha-TTS'))

# Core imports
import json
import numpy as np
import torch
import torchaudio
import librosa
from datetime import datetime
from dataclasses import dataclass
from typing import Dict, Optional
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


In [13]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class EmotionClassifierOutput:
    """
    Output from RESILIENT emotion classifier.
    
    Attributes:
        response: Therapy response text
        emotion: Discrete emotion label
        valence: -1 (negative) to +1 (positive)
        arousal: 0 (calm) to 1 (excited)
        dominance: 0 (submissive) to 1 (dominant)
        confidence: 0 to 1 (model confidence)
        therapy: Therapy type (CBT, MI, ACT, Supportive, etc.)
    """
    response: str
    emotion: str
    valence: float
    arousal: float
    dominance: float
    confidence: float
    therapy: str

class InputValidator:
    """Validates emotion classifier output"""
    
    VALID_EMOTIONS = {
        'Hopeful', 'Sad', 'Anxious', 'Proud', 'Ashamed', 
        'Angry', 'Neutral', 'Depressed', 'Guilty', 'Relieved'
    }
    
    VALID_THERAPIES = {
        'CBT', 'MI', 'DBT', 'ACT', 'Psychoeducation', 'Supportive'
    }
    
    @staticmethod
    def validate(data: Dict) -> EmotionClassifierOutput:
        """
        Validate and parse emotion classifier output.
        
        Args:
            data: Dictionary with keys: response, emotion, valence, arousal, dominance, confidence, therapy
            
        Returns:
            EmotionClassifierOutput or raises ValueError
        """
        
        # Validate required fields
        required_fields = ['response', 'emotion', 'valence', 'arousal', 'dominance', 'confidence', 'therapy']
        missing = [f for f in required_fields if f not in data]
        
        if missing:
            raise ValueError(f"Missing required fields: {missing}")
        
        # Extract and validate
        response = str(data['response']).strip()
        emotion = str(data['emotion']).strip()
        therapy = str(data['therapy']).strip()
        
        if not response:
            raise ValueError("Response text cannot be empty")
        
        if emotion not in InputValidator.VALID_EMOTIONS:
            logger.warning(f"Emotion '{emotion}' not in valid set, using 'Neutral'")
            emotion = 'Neutral'
        
        if therapy not in InputValidator.VALID_THERAPIES:
            logger.warning(f"Therapy '{therapy}' not in valid set, using 'Supportive'")
            therapy = 'Supportive'
        
        # Clamp values to valid ranges
        valence = float(np.clip(data['valence'], -1.0, 1.0))
        arousal = float(np.clip(data['arousal'], 0.0, 1.0))
        dominance = float(np.clip(data['dominance'], 0.0, 1.0))
        confidence = float(np.clip(data['confidence'], 0.0, 1.0))
        
        return EmotionClassifierOutput(
            response=response,
            emotion=emotion,
            valence=valence,
            arousal=arousal,
            dominance=dominance,
            confidence=confidence,
            therapy=therapy
        )

logger.info("✓ Data structures and validators initialized")

2026-07-07 13:25:30,121 INFO ✓ Data structures and validators initialized


In [73]:
# class InstructionGenerator:
#     """
#     Generates natural language instructions for CosyVoice from VAD+emotion+therapy.
    
#     Uses rich English descriptions without XML tags.
#     Instructions are appended with <|endofprompt|> token.
#     """
    
#     # Therapy-specific base personas
#     THERAPY_PERSONAS = {
#         'CBT': {
#             'role': 'professional cognitive behavioral therapist',
#             'style_words': ['structured', 'analytical', 'clear', 'logical'],
#             'tone': 'encouraging yet professional'
#         },
#         'MI': {
#             'role': 'compassionate motivational interviewer',
#             'style_words': ['collaborative', 'respectful', 'non-judgmental', 'affirming'],
#             'tone': 'warm and supportive'
#         },
#         'ACT': {
#             'role': 'mindful acceptance and commitment therapist',
#             'style_words': ['present', 'accepting', 'calm', 'centered'],
#             'tone': 'peaceful and grounded'
#         },
#         'DBT': {
#             'role': 'validating dialectical behavior therapist',
#             'style_words': ['validating', 'skills-focused', 'steady', 'practical'],
#             'tone': 'grounded and supportive'
#         },
#         'Psychoeducation': {
#             'role': 'knowledgeable educator',
#             'style_words': ['informative', 'clear', 'accessible', 'educational'],
#             'tone': 'expert yet approachable'
#         },
#         'Supportive': {
#             'role': 'warm and empathetic counselor',
#             'style_words': ['caring', 'empathetic', 'gentle', 'patient'],
#             'tone': 'compassionate and accepting'
#         }
#     }
    
#     # Valence-specific emotional descriptions
#     VALENCE_DESCRIPTIONS = {
#         'high_positive': {
#             'emotional_state': 'uplifted and encouraging',
#             'energy_level': 'energetic but grounded',
#             'message': 'convey hope and positive direction'
#         },
#         'moderate_positive': {
#             'emotional_state': 'warm and reassuring',
#             'energy_level': 'calm and steady',
#             'message': 'inspire confidence and support'
#         },
#         'neutral': {
#             'emotional_state': 'balanced and measured',
#             'energy_level': 'neutral and controlled',
#             'message': 'provide clear objective information'
#         },
#         'moderate_negative': {
#             'emotional_state': 'compassionate and reflective',
#             'energy_level': 'calm and steady',
#             'message': 'demonstrate deep understanding and validation'
#         },
#         'high_negative': {
#             'emotional_state': 'deeply empathetic and understanding',
#             'energy_level': 'gentle and patient',
#             'message': 'show profound compassion for their struggle'
#         }
#     }
    
#     @staticmethod
#     def _get_valence_category(valence: float) -> str:
#         """Map valence score to category"""
#         if valence > 0.5:
#             return 'high_positive'
#         elif valence > 0.1:
#             return 'moderate_positive'
#         elif valence > -0.1:
#             return 'neutral'
#         elif valence > -0.5:
#             return 'moderate_negative'
#         else:
#             return 'high_negative'
    
#     @staticmethod
#     def _arousal_to_pacing(arousal: float) -> str:
#         """Convert arousal to speaking pace description"""
#         if arousal > 0.7:
#             return 'at a slightly faster pace, with energy and engagement'
#         elif arousal > 0.4:
#             return 'at a normal, steady pace'
#         else:
#             return 'at a slower, deliberate pace'
    
#     @staticmethod
#     def _dominance_to_confidence(dominance: float) -> str:
#         """Convert dominance to confidence level description"""
#         if dominance > 0.7:
#             return 'with confident authority'
#         elif dominance > 0.4:
#             return 'with assured confidence'
#         else:
#             return 'gently and carefully'
    
#     @staticmethod
#     def _confidence_to_intensity(confidence: float) -> str:
#         """Map model confidence to instruction intensity"""
#         if confidence > 0.85:
#             return 'strong'
#         elif confidence > 0.70:
#             return 'moderate'
#         else:
#             return 'subtle'
    
#     @classmethod
#     def generate_instruction(cls,
#                             emotion: str,
#                             valence: float,
#                             arousal: float,
#                             dominance: float,
#                             confidence: float,
#                             therapy: str) -> str:
#         """
#         Generate natural language instruction for CosyVoice.
        
#         Args:
#             emotion: Emotion label (Hopeful, Sad, Anxious, etc.)
#             valence: -1 (negative) to +1 (positive)
#             arousal: 0 (calm) to 1 (excited)
#             dominance: 0 (submissive) to 1 (dominant)
#             confidence: 0 to 1 (model confidence in prediction)
#             therapy: Therapy type (CBT, MI, ACT, etc.)
            
#         Returns:
#             Natural language instruction with <|endofprompt|> token
#         """
        
#         # Get therapy persona
#         persona = cls.THERAPY_PERSONAS.get(therapy, cls.THERAPY_PERSONAS['Supportive'])
        
#         # Get valence description
#         valence_cat = cls._get_valence_category(valence)
#         valence_desc = cls.VALENCE_DESCRIPTIONS[valence_cat]
        
#         # Get arousal-based pacing
#         pacing = cls._arousal_to_pacing(arousal)
        
#         # Get dominance-based confidence expression
#         confidence_expr = cls._dominance_to_confidence(dominance)
        
#         # Get intensity based on model confidence
#         intensity = cls._confidence_to_intensity(confidence)
        
#         # Build the instruction
#         instruction = f"""You are a {persona['role']}.

# Your emotional state: {valence_desc['emotional_state']}.

# Speak {confidence_expr}.

# Communicate {pacing}.

# Your energy level: {valence_desc['energy_level']}.

# Your goal: {valence_desc['message']}.

# Use a {intensity} emotional delivery.

# Your tone should be {persona['tone']}.

# Maintain a therapeutic presence.

# Be authentic and genuine in your delivery.

# Avoid sounding robotic or artificial.

# Connect emotionally with the listener while maintaining professional boundaries."""
        
#         # Append CosyVoice instruction terminator
#         instruction += "\n<|endofprompt|>"
        
#         return instruction

# # Test the instruction generator
# logger.info("✓ Instruction generator initialized")

class InstructionGenerator:
    """
    Generates short expressive instructions for CosyVoice Instruct.

    The instruction controls ONLY speaking style.
    The response text already contains the therapeutic content.
    """

    # Base speaking style for each detected emotion
    BASE_STYLE = {
        "Hopeful": "Speak warmly with gentle optimism",
        "Happy": "Speak cheerfully with relaxed positivity",
        "Sad": "Speak softly with deep compassion",
        "Anxious": "Speak slowly with reassurance",
        "Fearful": "Speak gently with comfort",
        "Angry": "Speak firmly while remaining calm",
        "Neutral": "Speak naturally with a balanced tone",
        "Supportive": "Speak warmly with empathy",
        "Calm": "Speak calmly with reassurance",
        "Concerned": "Speak gently with concern and empathy"
    }

    @staticmethod
    def _valence_modifier(valence: float) -> str:
        """Modify emotional warmth."""

        if valence >= 0.6:
            return "using optimistic warmth"

        elif valence >= 0.2:
            return "using a warm tone"

        elif valence >= -0.2:
            return "using a balanced tone"

        elif valence >= -0.6:
            return "using compassionate warmth"

        else:
            return "using deep empathy"

    @staticmethod
    def _arousal_modifier(arousal: float) -> str:
        """Modify speaking pace."""

        if arousal >= 0.7:
            return "at an energetic pace"

        elif arousal >= 0.4:
            return "at a natural pace"

        else:
            return "at a slow and calming pace"

    @staticmethod
    def _dominance_modifier(dominance: float) -> str:
        """Modify confidence of delivery."""

        if dominance >= 0.7:
            return "with confident delivery"

        elif dominance >= 0.4:
            return "with steady confidence"

        else:
            return "with gentle reassurance"

    @staticmethod
    def _confidence_modifier(confidence: float) -> str:
        """
        If classifier confidence is low,
        avoid making the voice overly expressive.
        """

        if confidence >= 0.90:
            return ""

        elif confidence >= 0.70:
            return "naturally"

        else:
            return "subtly"

    @classmethod
    def generate_instruction(
        cls,
        emotion: str,
        valence: float,
        arousal: float,
        dominance: float,
        confidence: float,
        therapy: str = None
    ) -> str:
        """
        Generate a concise speaking instruction.

        Returns
        -------
        Example:
        Speak warmly with gentle optimism,
        using optimistic warmth,
        with steady confidence,
        at a natural pace.
        """

        base = cls.BASE_STYLE.get(
            emotion,
            cls.BASE_STYLE["Supportive"]
        )

        parts = [base]

        confidence_word = cls._confidence_modifier(confidence)

        if confidence_word:
            parts.append(confidence_word)

        parts.append(cls._valence_modifier(valence))
        parts.append(cls._dominance_modifier(dominance))
        parts.append(cls._arousal_modifier(arousal))

        instruction = ", ".join(parts) + "."
        instruction += "<|endofprompt|>"

        return instruction


logger.info("✓ Instruction generator initialized")

2026-07-07 17:28:25,588 INFO ✓ Instruction generator initialized


In [74]:
class SpeakerReferenceManager:
    """Manages speaker reference audio files for voice cloning"""
    
    TARGET_SR = 16000  # CosyVoice expects 16kHz
    
    def __init__(self, reference_dir: str = 'speaker_references'):
        """
        Initialize speaker reference manager.
        
        Args:
            reference_dir: Directory containing .wav files
                          Expected format: {speaker_id}.wav
        """
        self.reference_dir = Path(reference_dir)
        self.reference_dir.mkdir(exist_ok=True, parents=True)
        
        self.references = {}
        self._load_references()
    
    def _load_references(self):
        """Load all .wav files from reference directory"""
        
        wav_files = list(self.reference_dir.glob('*.wav'))
        
        if not wav_files:
            logger.warning(f"No reference audio files found in {self.reference_dir}")
            return
        
        for wav_file in wav_files:
            try:
                speaker_id = wav_file.stem
                
                # Load at 16kHz (required by CosyVoice)
                audio, sr = librosa.load(str(wav_file), sr=self.TARGET_SR)
                
                # Validate audio
                duration = len(audio) / sr
                if duration < 3.0 or duration > 10.0:
                    logger.warning(f"Reference {speaker_id} duration {duration:.1f}s, expected 3-10s")
                
                self.references[speaker_id] = audio
                logger.info(f"Loaded reference: {speaker_id} ({duration:.1f}s, {sr}Hz)")
            
            except Exception as e:
                logger.error(f"Failed to load {wav_file.name}: {e}")
    
    def get_reference(self, speaker_id: str = None) -> np.ndarray:
        """
        Get speaker reference audio.
        
        Args:
            speaker_id: Speaker identifier. If None, returns first available.
            
        Returns:
            Audio waveform (16kHz, mono, numpy array)
            
        Raises:
            ValueError if speaker not found
        """
        
        if not self.references:
            raise ValueError("No speaker references loaded")
        
        if speaker_id is None:
            speaker_id = list(self.references.keys())[0]
            logger.info(f"No speaker specified, using {speaker_id}")
        
        if speaker_id not in self.references:
            available = list(self.references.keys())
            raise ValueError(f"Speaker '{speaker_id}' not found. Available: {available}")
        
        return self.references[speaker_id]
    
    def list_speakers(self) -> list:
        """List all available speakers"""
        return list(self.references.keys())

logger.info("✓ Speaker reference manager initialized")

2026-07-07 17:28:30,704 INFO ✓ Speaker reference manager initialized


In [ ]:
# from pathlib import Path
# from cosyvoice.cli.cosyvoice import AutoModel

# logger.info("=" * 70)
# logger.info("Loading CosyVoice model...")
# logger.info("=" * 70)

# # ------------------------------------------------------------------
# # Prefer a local model if it already exists.
# # Otherwise let AutoModel download it automatically via ModelScope.
# # ------------------------------------------------------------------

# LOCAL_MODEL = Path("pretrained_models/CosyVoice-300M-Instruct")

# if LOCAL_MODEL.exists():
#     model_source = str(LOCAL_MODEL)
#     logger.info(f"Using local model: {model_source}")
# else:
#     model_source = "iic/CosyVoice-300M-Instruct"
#     logger.info("Local model not found.")
#     logger.info("AutoModel will download it from ModelScope...")

# try:
#     cosyvoice = AutoModel(
#         model_dir=model_source,
#         load_jit=True,
#         fp16=True
#     )

#     logger.info("✓ CosyVoice model loaded successfully")
#     logger.info(f"Sample Rate : {cosyvoice.sample_rate}")

#     # Verify the available speakers
#     speakers = cosyvoice.list_available_spks()
#     logger.info(f"Available Speakers ({len(speakers)}):")
#     for spk in speakers:
#         logger.info(f"  • {spk}")

# except Exception as e:
#     logger.exception("Failed to load CosyVoice model.")
#     raise


from pathlib import Path
from cosyvoice.cli.cosyvoice import AutoModel

# ==========================================================
# CELL 9 : Load Local CosyVoice Model
# ==========================================================

model_dir = Path("pretrained_models/CosyVoice-300M-Instruct")

if not model_dir.exists():
    raise FileNotFoundError(
        f"""
Model not found!

Expected location:

{model_dir}

Please copy the downloaded ModelScope model here.
"""
    )

logger.info("=" * 70)
logger.info("Loading CosyVoice model...")
logger.info("=" * 70)

# Apple Silicon / CPU
cosyvoice = AutoModel(
    model_dir=str(model_dir),
    load_jit=False,
    load_trt=False,
    fp16=False
)

logger.info("✓ CosyVoice model loaded successfully")
logger.info(f"Sample Rate : {cosyvoice.sample_rate}")

available_speakers = cosyvoice.list_available_spks()

logger.info(f"Available Speakers ({len(available_speakers)}):")

for spk in available_speakers:
    logger.info(f"    • {spk}")

2026-07-07 14:54:58,393 INFO ======================================================================
2026-07-07 14:54:58,397 INFO Loading CosyVoice model...
2026-07-07 14:54:58,398 INFO ======================================================================
2026-07-07 14:54:58,399 INFO Local model not found.
2026-07-07 14:54:58,399 INFO AutoModel will download it from ModelScope...
2026-07-07 14:54:58,414 DEBUG Starting new HTTPS connection (1): www.modelscope.cn:443


2026-07-07 14:54:59,683 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/iic/CosyVoice-300M-Instruct/revisions HTTP/1.1" 200 222
2026-07-07 14:55:00,042 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/iic/CosyVoice-300M-Instruct/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None
/Users/arshgarg/Drug_Deepfake/venv/lib/python3.11/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2026-07-07 14:55:04,213 INFO input frame rate=50
2026-07-07 14:55:07,772 DEBUG Starting new HTTPS connection (1): www.modelscope.cn:443


2026-07-07 14:55:08,905 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/revisions HTTP/1.1" 200 222
2026-07-07 14:55:09,269 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None
2026-07-07 14:55:10,370 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/repo?Revision=master&FilePath=configuration.json HTTP/1.1" 200 36
2026-07-07 14:55:11,433 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/repo?Revision=master&FilePath=en_tn_tagger.fst HTTP/1.1" 302 342
2026-07-07 14:55:11,436 DEBUG Starting new HTTPS connection (1): cdn-lfs-cn-1.modelscope.cn:443
2026-07-07 14:55:12,877 DEBUG https://cdn-lfs-cn-1.modelscope.cn:443 "GET /prod/lfs-objects/53/c6/02049b21c8041c6c1e2e28315315e236eee2944fbab059f46ee3654d51bd?filename=en_tn_tagger.fst&namespace=pengzhendong&repository=wetext&revision=master&tag=model&auth_key=1783416311-5cc55c6d897c

2026-07-07 14:55:57,402 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/revisions HTTP/1.1" 200 222
2026-07-07 14:55:57,730 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None
2026-07-07 14:55:58,130 INFO use wetext frontend
2026-07-07 14:55:58,131 WARNING no cuda device, set load_jit/load_trt/fp16 to False
2026-07-07 14:56:04,730 INFO ✓ CosyVoice model loaded successfully
2026-07-07 14:56:04,733 INFO Sample Rate : 22050
2026-07-07 14:56:04,734 INFO Available Speakers (7):
2026-07-07 14:56:04,735 INFO   • 中文女
2026-07-07 14:56:04,735 INFO   • 中文男
2026-07-07 14:56:04,736 INFO   • 日语男
2026-07-07 14:56:04,736 INFO   • 粤语女
2026-07-07 14:56:04,736 INFO   • 英文女
2026-07-07 14:56:04,736 INFO   • 英文男
2026-07-07 14:56:04,737 INFO   • 韩语女


In [75]:
class AudioPostProcessor:
    """Post-processes synthesized audio for quality improvement"""
    
    def __init__(self, target_sr: int = 22050):
        """
        Initialize post-processor.
        
        Args:
            target_sr: Target sample rate (Hz)
        """
        self.target_sr = target_sr
    
    def process(self, 
                waveform: torch.Tensor,
                input_sr: int) -> torch.Tensor:
        """
        Apply post-processing to audio.
        
        Steps:
        1. Normalize loudness to -20 LUFS
        2. Apply fade-in/fade-out
        3. Prevent clipping
        
        Args:
            waveform: Input audio waveform (torch tensor)
            input_sr: Input sample rate
            
        Returns:
            Processed waveform
        """
        
        # Ensure float32
        if waveform.dtype != torch.float32:
            waveform = waveform.float()
        
        # Normalize loudness
        rms = torch.sqrt(torch.mean(waveform ** 2))
        
        if rms > 1e-6:
            target_rms = 0.1  # -20 dB
            gain = target_rms / rms
            waveform = waveform * gain
        
        # Soft clipping to prevent distortion
        threshold = 0.95
        waveform = torch.where(
            torch.abs(waveform) > threshold,
            torch.sign(waveform) * (threshold + (1.0 - threshold) * torch.tanh(
                (torch.abs(waveform) - threshold) / (1.0 - threshold)
            )),
            waveform
        )
        
        # Apply fades
        fade_duration_ms = 50
        fade_samples = int(fade_duration_ms * input_sr / 1000)
        
        if fade_samples > 0 and waveform.shape[-1] > fade_samples * 2:
            # Fade-in
            fade_in = torch.linspace(0, 1, fade_samples)
            waveform[..., :fade_samples] *= fade_in
            
            # Fade-out
            fade_out = torch.linspace(1, 0, fade_samples)
            waveform[..., -fade_samples:] *= fade_out
        
        # Hard clip to [-1, 1]
        waveform = torch.clamp(waveform, -1.0, 1.0)
        
        return waveform

logger.info("✓ Audio post-processor initialized")

2026-07-07 17:28:44,511 INFO ✓ Audio post-processor initialized


In [76]:
class SafetyGate:
    """Final safety validation before audio delivery"""
    
    CRISIS_KEYWORDS = {
        'suicide', 'kill myself', 'kill myself', 'end it',
        'don\'t want to live', 'harm myself', 'hurt myself',
        'overdose', 'poison myself'
    }
    
    THERAPEUTIC_RED_FLAGS = {
        'you\'re hopeless',
        'you\'ll never recover',
        'just give up',
        'it\'s impossible',
        'there\'s no point'
    }
    
    @staticmethod
    def validate(response_text: str, 
                 waveform: torch.Tensor,
                 sample_rate: int) -> tuple:
        """
        Perform safety validation.
        
        Args:
            response_text: Original response text
            waveform: Synthesized audio
            sample_rate: Audio sample rate
            
        Returns:
            (passed: bool, report: dict)
        """
        
        report = {
            'passed': True,
            'messages': [],
            'checks': {}
        }
        
        # Check 1: Crisis keywords
        text_lower = response_text.lower()
        crisis_found = any(keyword in text_lower for keyword in SafetyGate.CRISIS_KEYWORDS)
        
        if crisis_found:
            report['passed'] = False
            report['messages'].append('✗ CRITICAL: Crisis content detected')
            report['checks']['crisis'] = False
        else:
            report['messages'].append('✓ No crisis keywords detected')
            report['checks']['crisis'] = True
        
        # Check 2: Therapeutic red flags
        flags_found = any(flag in text_lower for flag in SafetyGate.THERAPEUTIC_RED_FLAGS)
        
        if flags_found:
            report['passed'] = False
            report['messages'].append('✗ Therapeutic red flag detected')
            report['checks']['therapeutic'] = False
        else:
            report['messages'].append('✓ Therapeutic content appropriate')
            report['checks']['therapeutic'] = True
        
        # Check 3: Audio integrity
        duration = len(waveform[0]) / sample_rate if len(waveform.shape) > 1 else len(waveform) / sample_rate
        
        if duration < 0.5 or duration > 120:
            report['passed'] = False
            report['messages'].append(f'✗ Duration {duration:.1f}s outside valid range [0.5, 120]')
            report['checks']['duration'] = False
        else:
            report['messages'].append(f'✓ Duration valid ({duration:.1f}s)')
            report['checks']['duration'] = True
        
        # Check 4: Clipping
        max_val = torch.max(torch.abs(waveform))
        if max_val > 0.99:
            report['messages'].append(f'⚠ Audio near clipping threshold ({max_val:.3f})')
            report['checks']['clipping'] = False
        else:
            report['messages'].append('✓ No clipping detected')
            report['checks']['clipping'] = True
        
        return report['passed'], report

logger.info("✓ Safety gate initialized")

2026-07-07 17:28:51,863 INFO ✓ Safety gate initialized


In [77]:
class EmotionAwareTTSPipeline:
    """
    Complete emotion-aware speech synthesis pipeline.
    
    Processes emotion classifier output → natural language instruction → 
    CosyVoice synthesis → post-processed audio
    """
    
    def __init__(self,
                 model,
                 speaker_manager: SpeakerReferenceManager,
                 output_dir: str = 'output'):
        """
        Initialize pipeline.
        
        Args:
            model: Loaded CosyVoice AutoModel instance
            speaker_manager: SpeakerReferenceManager instance
            output_dir: Directory to save output audio
        """
        
        self.model = model
        self.speaker_manager = speaker_manager
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True, parents=True)
        
        self.instruction_generator = InstructionGenerator()
        self.post_processor = AudioPostProcessor(target_sr=model.sample_rate)
        self.safety_gate = SafetyGate()
        
        self.validator = InputValidator()
    
    def process(self,
                classifier_output: Dict,
                speaker_id: str = None,
                save_audio: bool = True) -> Dict:
        """
        Process emotion classifier output to generate emotion-aware speech.
        
        Args:
            classifier_output: Dict with keys:
                - response: Therapy response text
                - emotion: Emotion label
                - valence: Valence score
                - arousal: Arousal score
                - dominance: Dominance score
                - confidence: Model confidence
                - therapy: Therapy type
            
            speaker_id: Speaker to use for voice cloning (optional)
            save_audio: Whether to save output to file
            
        Returns:
            Dict with:
                - success: bool
                - audio: torch.Tensor (if successful)
                - path: str (file path if saved)
                - metadata: dict
                - errors: list
        """
        
        result = {
            'success': False,
            'audio': None,
            'path': None,
            'metadata': {},
            'errors': []
        }
        
        try:
            logger.info("="*70)
            logger.info("EMOTION-AWARE TTS PIPELINE")
            logger.info("="*70)
            
            # Stage 1: Validate input
            logger.info("\n[1/6] Validating input...")
            emotion_data = self.validator.validate(classifier_output)
            logger.info(f"  ✓ Valid input: {emotion_data.emotion} (valence={emotion_data.valence:.2f})")
            
            # Stage 2: Generate instruction
            logger.info("\n[2/6] Generating natural language instruction...")
            instruction = self.instruction_generator.generate_instruction(
                emotion=emotion_data.emotion,
                valence=emotion_data.valence,
                arousal=emotion_data.arousal,
                dominance=emotion_data.dominance,
                confidence=emotion_data.confidence,
                therapy=emotion_data.therapy
            )
            logger.info(f"  ✓ Instruction generated ({len(instruction)} chars)")
            
            # # Stage 3: Load speaker reference
            # logger.info("\n[3/6] Loading speaker reference...")
            # speaker_ref = self.speaker_manager.get_reference(speaker_id)
            # logger.info(f"  ✓ Speaker reference loaded ({len(speaker_ref)/16000:.1f}s)")
            
            # # Stage 4: Synthesize with CosyVoice
            # logger.info("\n[4/6] Synthesizing speech with CosyVoice...")
            
            # # CosyVoice inference using latest AutoModel API
            # audio_output = None
            
            # for i, output in enumerate(self.model.inference_instruct(
            #     text=emotion_data.response,
            #     instruct_text=instruction,
            #     stream=False
            # )):
            #     audio_output = output['tts_speech']

            # -------------------------------------------------------
        # Stage 3: Select speaker
        # -------------------------------------------------------
            logger.info("\n[3/6] Selecting speaker...")

            available_speakers = self.model.list_available_spks()

            if speaker_id is None:
                speaker_id = "英文男"

            if speaker_id not in available_speakers:
                logger.warning(
                    f"Speaker '{speaker_id}' not found. "
                    f"Using '{available_speakers[0]}' instead."
                )
                speaker_id = available_speakers[0]

            logger.info(f"  ✓ Using speaker: {speaker_id}")

        # -------------------------------------------------------
        # Stage 4: CosyVoice Inference
        # -------------------------------------------------------
            logger.info("\n[4/6] Synthesizing speech with CosyVoice...")

            audio_output = None

            for output in self.model.inference_instruct(
                tts_text=emotion_data.response,
                spk_id=speaker_id,
                instruct_text=instruction,
                stream=False
            ):
            
                audio_output = output["tts_speech"]
            
            if audio_output is None:
                raise ValueError("CosyVoice synthesis returned no audio")
            
            # Ensure tensor format
            # if isinstance(audio_output, np.ndarray):
            #     audio_output = torch.from_numpy(audio_output)
            if isinstance(audio_output, np.ndarray):
                audio_output = torch.from_numpy(audio_output)

            if audio_output.ndim == 1:
                audio_output = audio_output.unsqueeze(0)

            audio_output = audio_output.float()
            
            duration = len(audio_output[0]) / self.model.sample_rate if len(audio_output.shape) > 1 else len(audio_output) / self.model.sample_rate
            logger.info(f"  ✓ Synthesis complete ({duration:.2f}s)")
            
            # Stage 5: Post-process audio
            logger.info("\n[5/6] Post-processing audio...")
            audio_processed = self.post_processor.process(audio_output, self.model.sample_rate)
            logger.info(f"  ✓ Post-processing complete")
            
            # Stage 6: Safety validation
            logger.info("\n[6/6] Running safety validation...")
            passed, safety_report = self.safety_gate.validate(
                emotion_data.response,
                audio_processed,
                self.model.sample_rate
            )
            
            for msg in safety_report['messages']:
                logger.info(f"  {msg}")
            
            if not passed:
                result['errors'].append("Safety validation failed")
                logger.error("✗ Audio rejected by safety gate")
                return result
            
            # Success
            result['success'] = True
            result['audio'] = audio_processed
            result['metadata'] = {
                'response': emotion_data.response,
                'emotion': emotion_data.emotion,
                'valence': emotion_data.valence,
                'arousal': emotion_data.arousal,
                'dominance': emotion_data.dominance,
                'confidence': emotion_data.confidence,
                'therapy': emotion_data.therapy,
                'duration': float(duration),
                'sample_rate': self.model.sample_rate,
                'instruction': instruction
            }
            
            # Save audio if requested
            if save_audio:
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                filename = f"response_{timestamp}_{emotion_data.emotion}.wav"
                output_path = self.output_dir / filename

                audio_np = audio_processed.squeeze().cpu().numpy()
                import soundfile as sf
                sf.write(
                    str(output_path),
                    audio_np,
                    self.model.sample_rate
                )
                
                result['path'] = str(output_path)
                logger.info(f"  ✓ Audio saved: {filename}")
            
            logger.info("\n" + "="*70)
            logger.info("✓ SYNTHESIS COMPLETE")
            logger.info("="*70 + "\n")
            
            return result
        
        except Exception as e:
            logger.error(f"Pipeline error: {e}")
            import traceback
            traceback.print_exc()
            result['errors'].append(str(e))
            return result

# Initialize pipeline components
speaker_manager = SpeakerReferenceManager('speaker_references')
pipeline = EmotionAwareTTSPipeline(cosyvoice, speaker_manager)

logger.info("✓ Complete pipeline initialized and ready")

2026-07-07 17:28:53,685 WARNING No reference audio files found in speaker_references
2026-07-07 17:28:53,686 INFO ✓ Complete pipeline initialized and ready


In [80]:
sample_input = {
    "response": (
        "I can see this has been really difficult for you. "
        "That's completely understandable. "
        "Let's break this down into smaller manageable steps "
        "so you can regain a sense of control."
    ),

    "emotion": "Anxious",
    "valence": -0.45,
    "arousal": 0.25,
    "dominance": 0.20,
    "confidence": 0.94,
    "therapy": "CBT"
}

result = pipeline.process(
    classifier_output=sample_input,
    speaker_id="英文男",      # Built-in English male
    save_audio=True
)

print("\n==============================")

if result["success"]:
    print("SUCCESS")
    print(result["path"])
    print(result["metadata"])

else:
    print(result["errors"])

2026-07-07 17:39:20,555 INFO ======================================================================
2026-07-07 17:39:20,558 INFO EMOTION-AWARE TTS PIPELINE
2026-07-07 17:39:20,558 INFO ======================================================================
2026-07-07 17:39:20,560 INFO 
[1/6] Validating input...
2026-07-07 17:39:20,567 INFO   ✓ Valid input: Anxious (valence=-0.45)
2026-07-07 17:39:20,567 INFO 
[2/6] Generating natural language instruction...
2026-07-07 17:39:20,568 INFO   ✓ Instruction generated (126 chars)
2026-07-07 17:39:20,568 INFO 
[3/6] Selecting speaker...
2026-07-07 17:39:20,569 INFO   ✓ Using speaker: 英文男
2026-07-07 17:39:20,569 INFO 
[4/6] Synthesizing speech with CosyVoice...
  0%|          | 0/1 [00:00<?, ?it/s]2026-07-07 17:39:20,604 INFO synthesis text I can see this has been really difficult for you. That's completely understandable. Let's break this down into smaller manageable steps so you can regain a sense of control.
2026-07-07 17:40:20,558 INFO yield


SUCCESS
output/response_20260707_174020_Anxious.wav
{'response': "I can see this has been really difficult for you. That's completely understandable. Let's break this down into smaller manageable steps so you can regain a sense of control.", 'emotion': 'Anxious', 'valence': -0.45, 'arousal': 0.25, 'dominance': 0.2, 'confidence': 0.94, 'therapy': 'CBT', 'duration': 11.679637188208616, 'sample_rate': 22050, 'instruction': 'Speak slowly with reassurance, using compassionate warmth, with gentle reassurance, at a slow and calming pace.<|endofprompt|>'}


In [86]:
import soundfile as sf
import torch

REFERENCE_WAV = "/Users/arshgarg/Drug_Deepfake/speaker_references/test.wav"

PROMPT_TEXT = (
    "are the biggest up's and down's that have come in your journey to recovery."
)

TARGET_TEXT = (
    "I can see this has been really difficult for you. "
    "That's completely understandable. "
    "Let's break this down into smaller manageable steps "
    "so you can regain a sense of control."
)

audio = None

print("Running Zero-Shot Test...")

for output in cosyvoice.inference_zero_shot(
    tts_text=TARGET_TEXT,
    prompt_text=PROMPT_TEXT,
    prompt_wav=REFERENCE_WAV,
    stream=False
):
    audio = output["tts_speech"]

if isinstance(audio, torch.Tensor):
    audio = audio.squeeze().cpu().numpy()

sf.write(
    "zero_shot_test.wav",
    audio,
    cosyvoice.sample_rate
)

print("✓ Saved as zero_shot_test.wav")

Running Zero-Shot Test...


  0%|          | 0/1 [00:00<?, ?it/s]2026-07-07 22:00:39,635 INFO synthesis text I can see this has been really difficult for you. That's completely understandable. Let's break this down into smaller manageable steps so you can regain a sense of control.
2026-07-07 22:02:48,945 INFO yield speech len 8.359183673469389, rtf 15.469185652327722
100%|██████████| 1/1 [02:13<00:00, 133.47s/it]

✓ Saved as zero_shot_test.wav
